# 03 NPS Alert Data Collection
Notebook goal: Another repeat of previous notebook, then move to updated annotation data collection.



API documentation
https://www.nps.gov/subjects/developer/api-documentation.htm


## Updated annotation design

This version retains the original sampling and taxonomy plan but adjusts based on the instructor feedback.  The main task remains single-label classification. A separate optional secondary_label records multi-topic alerts. Raw alert text is preserved separately from cleaned text. Exact-duplicate text groups are retained so related records can later be assigned to the same data split. The pilot review will measure how often multi-topic and uncertain cases occur before the final taxonomy is frozen.


## 1. Add the API key to Google Colab Secrets




In [ ]:
from google.colab import userdata

NPS_API_KEY = userdata.get("NPS_API_KEY")

if not NPS_API_KEY:
    raise ValueError(
        "NPS_API_KEY was not found. Add it to Google Colab Secrets "
        "and enable notebook access."
    )

print("API key loaded successfully.")


API key loaded successfully.


## 2. Import packages and define the alerts endpoint

The first request will retrieve only five alerts so that the connection and response format can be checked before collecting more data.


In [ ]:
import json
import requests
import pandas as pd

BASE_URL = "https://developer.nps.gov/api/v1/alerts"

headers = {
    "X-Api-Key": NPS_API_KEY
}

params = {
    "limit": 5,
    "start": 0
}

print("Packages imported and request settings created.")


Packages imported and request settings created.


## 3. Make the initial API request




In [ ]:
try:
    response = requests.get(
        BASE_URL,
        headers=headers,
        params=params,
        timeout=30
    )

    print("Status code:", response.status_code)
    print("Request URL:", response.url)

    response.raise_for_status()

except requests.exceptions.Timeout as exc:
    raise RuntimeError("The NPS API request timed out.") from exc

except requests.exceptions.HTTPError as exc:
    raise RuntimeError(
        f"The NPS API returned an HTTP error: {response.status_code}\n"
        f"Response preview: {response.text[:500]}"
    ) from exc

except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"The NPS API request failed: {exc}") from exc


Status code: 200
Request URL: https://developer.nps.gov/api/v1/alerts?limit=5&start=0


## 4. Inspect the top-level response

This step checks the overall response structure and the number of alerts reported by the API.


In [ ]:
payload = response.json()

print("Top-level keys:", list(payload.keys()))
print("Reported total alerts:", payload.get("total"))
print("Number returned in this request:", len(payload.get("data", [])))


Top-level keys: ['total', 'limit', 'start', 'data']
Reported total alerts: 627
Number returned in this request: 5


## 5. Inspect the fields in the first alert

The project should use the fields that the API actually returns rather than assuming that every proposed field is available.


In [ ]:
alerts = payload.get("data", [])

if alerts:
    first_alert = alerts[0]

    print("Fields in one alert:")
    for field in first_alert.keys():
        print("-", field)
else:
    print("The request succeeded, but no alert records were returned.")


Fields in one alert:
- id
- url
- title
- parkCode
- description
- category
- relatedRoadEvents
- lastIndexedDate


## 6. View the complete first alert



In [ ]:
if alerts:
    print(json.dumps(alerts[0], indent=2))
else:
    print("No alert is available to display.")


{
  "id": "44ED09F6-52BC-4A67-9EE4-4D92DF0A75FB",
  "url": "",
  "title": "Road Closures 6/24 and 6/25 on Swift Brook and Katahdin Loop Roads",
  "parkCode": "kaww",
  "description": "On Wednesday, 6/24, and Thursday, 6/25, road closures will occur on Swift Brook Road at the bridge over Sandbank Stream and on the Loop Road just north of the junction with Wassataquoik Stream Road. Closures will last up to 1 hour while work is being done for bridge replacement projects.",
  "category": "Park Closure",
  "relatedRoadEvents": [],
  "lastIndexedDate": "2026-06-23 00:00:00.0"
}


## 7. Convert the sample to a pandas DataFrame


In [ ]:
sample_df = pd.DataFrame(alerts)

print("Shape:", sample_df.shape)
display(sample_df.head())


Shape: (5, 8)


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate
0,44ED09F6-52BC-4A67-9EE4-4D92DF0A75FB,,Road Closures 6/24 and 6/25 on Swift Brook and...,kaww,"On Wednesday, 6/24, and Thursday, 6/25, road c...",Park Closure,[],2026-06-23 00:00:00.0
1,FEDAF34D-B0F4-4399-9B06-14CBFFE89E42,,Temporary Closure of Pinnacle Road - June 25,cuga,"Pinnacle Road will be closed on Thursday, June...",Park Closure,[],2026-06-23 00:00:00.0
2,8DF8B523-E5C6-4E4C-AF5B-F641F8C3B6EE,https://camba.dualrates.com/a/r/szz/camba/trai...,Current East Rim Trail Status: Open,cuva,"As of Tuesday, June 16, 2026, the East Rim Tra...",Information,[],2026-06-23 00:00:00.0
3,F08A2A6E-5846-4829-B869-D51C077B067D,,Multiple Park Road Closures After Heavy Storm ...,vafo,Multiple park roads are closed due to downed t...,Park Closure,[{'title': 'Trees Down Due to Heavy Storm Wind...,2026-06-23 00:00:00.0
4,336BD9D6-432A-4F8F-A4FC-6EC2CFB285DE,,No Public Beach Access from Fort Moultrie duri...,fosu,There will be no public beach access from Fort...,Park Closure,[],2026-06-23 00:00:00.0


## 8. List the available columns


In [ ]:
print(sample_df.columns.tolist())


['id', 'url', 'title', 'parkCode', 'description', 'category', 'relatedRoadEvents', 'lastIndexedDate']


## 9. Display candidate project fields

This cell selects only candidate fields that are actually present, so it will not fail if the API omits one of the proposed fields.


In [ ]:
candidate_columns = [
    "id",
    "parkCode",
    "title",
    "description",
    "category",
    "url",
    "lastIndexedDate"
]

available_columns = [
    column
    for column in candidate_columns
    if column in sample_df.columns
]

print("Candidate fields available:", available_columns)

if available_columns:
    display(sample_df[available_columns])
else:
    print("None of the candidate columns were found.")


Candidate fields available: ['id', 'parkCode', 'title', 'description', 'category', 'url', 'lastIndexedDate']


,id,parkCode,title,description,category,url,lastIndexedDate
0,44ED09F6-52BC-4A67-9EE4-4D92DF0A75FB,kaww,Road Closures 6/24 and 6/25 on Swift Brook and...,"On Wednesday, 6/24, and Thursday, 6/25, road c...",Park Closure,,2026-06-23 00:00:00.0
1,FEDAF34D-B0F4-4399-9B06-14CBFFE89E42,cuga,Temporary Closure of Pinnacle Road - June 25,"Pinnacle Road will be closed on Thursday, June...",Park Closure,,2026-06-23 00:00:00.0
2,8DF8B523-E5C6-4E4C-AF5B-F641F8C3B6EE,cuva,Current East Rim Trail Status: Open,"As of Tuesday, June 16, 2026, the East Rim Tra...",Information,https://camba.dualrates.com/a/r/szz/camba/trai...,2026-06-23 00:00:00.0
3,F08A2A6E-5846-4829-B869-D51C077B067D,vafo,Multiple Park Road Closures After Heavy Storm ...,Multiple park roads are closed due to downed t...,Park Closure,,2026-06-23 00:00:00.0
4,336BD9D6-432A-4F8F-A4FC-6EC2CFB285DE,fosu,No Public Beach Access from Fort Moultrie duri...,There will be no public beach access from Fort...,Park Closure,,2026-06-23 00:00:00.0


## 10. Read the first five alerts in a traveler-friendly format



In [ ]:
if not alerts:
    print("No alerts were returned.")

for index, alert in enumerate(alerts, start=1):
    print("=" * 80)
    print(f"ALERT {index}")
    print("Park code:", alert.get("parkCode"))
    print("NPS category:", alert.get("category"))
    print("Title:", alert.get("title"))
    print("Description:", alert.get("description"))
    print("URL:", alert.get("url"))


ALERT 1
Park code: kaww
NPS category: Park Closure
Title: Road Closures 6/24 and 6/25 on Swift Brook and Katahdin Loop Roads
Description: On Wednesday, 6/24, and Thursday, 6/25, road closures will occur on Swift Brook Road at the bridge over Sandbank Stream and on the Loop Road just north of the junction with Wassataquoik Stream Road. Closures will last up to 1 hour while work is being done for bridge replacement projects.
URL: 
ALERT 2
Park code: cuga
NPS category: Park Closure
Title: Temporary Closure of Pinnacle Road - June 25
Description: Pinnacle Road will be closed on Thursday, June 25, from 8:00 a.m. to 2:30 p.m. for maintenance work.
URL: 
ALERT 3
Park code: cuva
NPS category: Information
Title: Current East Rim Trail Status: Open
Description: As of Tuesday, June 16, 2026, the East Rim Trail System remains open. For more information, visit the website linked below. In good conditions the mountain bike trails are open 6 am-11 pm for bikers and as posted for other trail users. In

# Collect and Inspect the Full Active-Alert Dataset

The five-record API test was successful. This section now:

1. retrieves all currently available alerts using pagination,
2. saves a dated raw JSON snapshot,
3. converts the alerts to a DataFrame,
4. checks missing values and duplicate IDs,
5. retrieves full park names from the NPS parks endpoint,
6. joins park names to the alert records, and
7. creates initial feasibility summaries.

Because active alerts change over time, the collection timestamp is saved with the data.


## 12. Define a reusable paginated API function

The function below repeatedly requests batches until it reaches the total number of records reported by the API.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import time

def collect_paginated_nps_data(
    endpoint,
    headers,
    extra_params=None,
    batch_size=50,
    pause_seconds=0.2
):
    """Collect all records from a paginated NPS API endpoint."""

    all_records = []
    start = 0
    reported_total = None
    extra_params = extra_params or {}

    while True:
        request_params = {
            **extra_params,
            "limit": batch_size,
            "start": start
        }

        response = requests.get(
            endpoint,
            headers=headers,
            params=request_params,
            timeout=30
        )
        response.raise_for_status()

        page = response.json()
        records = page.get("data", [])

        if reported_total is None:
            reported_total = int(page.get("total", 0))
            print(f"API reports {reported_total:,} total records.")

        all_records.extend(records)
        print(
            f"Collected {len(all_records):,} of "
            f"{reported_total:,} records."
        )

        if not records or len(all_records) >= reported_total:
            break

        start += len(records)
        time.sleep(pause_seconds)

    return all_records, reported_total


## 13. Collect all active alerts


In [ ]:
ALERTS_ENDPOINT = "https://developer.nps.gov/api/v1/alerts"

all_alerts, reported_alert_total = collect_paginated_nps_data(
    endpoint=ALERTS_ENDPOINT,
    headers=headers,
    batch_size=50
)

print("\nCollection complete.")
print("Reported total:", reported_alert_total)
print("Records collected:", len(all_alerts))


API reports 627 total records.
Collected 50 of 627 records.
Collected 100 of 627 records.
Collected 150 of 627 records.
Collected 200 of 627 records.
Collected 250 of 627 records.
Collected 300 of 627 records.
Collected 350 of 627 records.
Collected 400 of 627 records.
Collected 450 of 627 records.
Collected 500 of 627 records.
Collected 550 of 627 records.
Collected 600 of 627 records.
Collected 627 of 627 records.

Collection complete.
Reported total: 627
Records collected: 627


## 14. Create a collection timestamp and output folder


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import time
collection_time_utc = datetime.now(timezone.utc)
collection_timestamp = collection_time_utc.isoformat()
collection_date = collection_time_utc.strftime("%Y-%m-%d")
timestamp_for_filename = collection_time_utc.strftime("%Y%m%dT%H%M%SZ")

output_dir = Path("nps_alert_data")
output_dir.mkdir(exist_ok=True)

print("Collection timestamp:", collection_timestamp)
print("Output folder:", output_dir.resolve())


Collection timestamp: 2026-07-06T14:50:21.927326+00:00
Output folder: /content/nps_alert_data


## 15. Save the raw API response as JSON

This preserves the original API records before cleaning or transformation.


In [ ]:
raw_json_path = output_dir / f"nps_alerts_raw_{timestamp_for_filename}.json"

raw_snapshot = {
    "collection_timestamp_utc": collection_timestamp,
    "reported_total": reported_alert_total,
    "records_collected": len(all_alerts),
    "data": all_alerts
}

with raw_json_path.open("w", encoding="utf-8") as file:
    json.dump(raw_snapshot, file, indent=2, ensure_ascii=False)

print("Saved raw JSON:", raw_json_path)


Saved raw JSON: nps_alert_data/nps_alerts_raw_20260623T190917Z.json


## 16. Convert the complete alert collection to a DataFrame


In [ ]:
alerts_df = pd.DataFrame(all_alerts)

alerts_df["collection_timestamp_utc"] = collection_timestamp
alerts_df["collection_date"] = collection_date

print("Alert DataFrame shape:", alerts_df.shape)
print("\nColumns:")
print(alerts_df.columns.tolist())

display(alerts_df.head())


Alert DataFrame shape: (627, 10)

Columns:
['id', 'url', 'title', 'parkCode', 'description', 'category', 'relatedRoadEvents', 'lastIndexedDate', 'collection_timestamp_utc', 'collection_date']


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate,collection_timestamp_utc,collection_date
0,44ED09F6-52BC-4A67-9EE4-4D92DF0A75FB,,Road Closures 6/24 and 6/25 on Swift Brook and...,kaww,"On Wednesday, 6/24, and Thursday, 6/25, road c...",Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23
1,FEDAF34D-B0F4-4399-9B06-14CBFFE89E42,,Temporary Closure of Pinnacle Road - June 25,cuga,"Pinnacle Road will be closed on Thursday, June...",Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23
2,8DF8B523-E5C6-4E4C-AF5B-F641F8C3B6EE,https://camba.dualrates.com/a/r/szz/camba/trai...,Current East Rim Trail Status: Open,cuva,"As of Tuesday, June 16, 2026, the East Rim Tra...",Information,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23
3,F08A2A6E-5846-4829-B869-D51C077B067D,,Multiple Park Road Closures After Heavy Storm ...,vafo,Multiple park roads are closed due to downed t...,Park Closure,[{'title': 'Trees Down Due to Heavy Storm Wind...,2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23
4,336BD9D6-432A-4F8F-A4FC-6EC2CFB285DE,,No Public Beach Access from Fort Moultrie duri...,fosu,There will be no public beach access from Fort...,Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23


## 17. Save the initial structured CSV

This file keeps the API fields as returned, with only the collection timestamp added.


In [ ]:
initial_csv_path = (
    output_dir / f"nps_alerts_structured_{timestamp_for_filename}.csv"
)

alerts_df.to_csv(initial_csv_path, index=False)

print("Saved structured CSV:", initial_csv_path)


Saved structured CSV: nps_alert_data/nps_alerts_structured_20260623T190917Z.csv


## 18. Verify totals and unique alert IDs


In [ ]:
print("API-reported total:", reported_alert_total)
print("Rows collected:", len(alerts_df))

if "id" in alerts_df.columns:
    print("Unique alert IDs:", alerts_df["id"].nunique())
    print("Duplicate alert-ID rows:", alerts_df.duplicated(subset="id").sum())
else:
    print("No alert ID field was found.")


API-reported total: 627
Rows collected: 627
Unique alert IDs: 627
Duplicate alert-ID rows: 0


## 19. Inspect missing values in important fields


In [ ]:
important_fields = [
    "id",
    "parkCode",
    "title",
    "description",
    "category",
    "url",
    "lastIndexedDate"
]

available_important_fields = [
    field for field in important_fields
    if field in alerts_df.columns
]

missing_summary = pd.DataFrame({
    "field": available_important_fields,
    "missing_count": [
        alerts_df[field].isna().sum()
        + alerts_df[field].astype(str).str.strip().eq("").sum()
        for field in available_important_fields
    ]
})

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(alerts_df) * 100
).round(2)

display(missing_summary)


,field,missing_count,missing_percent
0,id,0,0.00
1,parkCode,1,0.16
2,title,0,0.00
3,description,0,0.00
4,category,0,0.00
5,url,345,55.02
6,lastIndexedDate,0,0.00


## 20. Inspect the original NPS alert-category distribution


In [ ]:
if "category" in alerts_df.columns:
    category_counts = (
        alerts_df["category"]
        .fillna("Missing")
        .replace("", "Missing")
        .value_counts()
        .rename_axis("nps_category")
        .reset_index(name="alert_count")
    )

    category_counts["percent"] = (
        category_counts["alert_count"] / len(alerts_df) * 100
    ).round(2)

    display(category_counts)
else:
    print("The category field is unavailable.")


,nps_category,alert_count,percent
0,Information,329,52.47
1,Park Closure,154,24.56
2,Caution,132,21.05
3,Danger,12,1.91


## 21. Measure alert-text length

The combined text will later support classification. These summaries help determine whether alerts are generally long enough to classify and whether a maximum transformer sequence length is needed.


In [ ]:
title_text = (
    alerts_df["title"].fillna("").astype(str)
    if "title" in alerts_df.columns
    else pd.Series("", index=alerts_df.index)
)

description_text = (
    alerts_df["description"].fillna("").astype(str)
    if "description" in alerts_df.columns
    else pd.Series("", index=alerts_df.index)
)

alerts_df["combined_text"] = (
    title_text.str.strip()
    + " "
    + description_text.str.strip()
).str.strip()

alerts_df["title_word_count"] = title_text.str.split().str.len()
alerts_df["description_word_count"] = description_text.str.split().str.len()
alerts_df["combined_word_count"] = alerts_df["combined_text"].str.split().str.len()
alerts_df["combined_character_count"] = alerts_df["combined_text"].str.len()

display(
    alerts_df[
        [
            "title_word_count",
            "description_word_count",
            "combined_word_count",
            "combined_character_count"
        ]
    ].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

,title_word_count,description_word_count,combined_word_count,combined_character_count
count,627.000000,627.000000,627.000000,627.000000
mean,6.516746,37.913876,44.430622,274.513557
std,3.087275,15.723212,16.550037,101.957868
min,1.000000,2.000000,4.000000,21.000000
25%,4.000000,27.000000,33.000000,206.000000
50%,6.000000,37.000000,43.000000,268.000000
75%,9.000000,44.000000,52.000000,313.000000
90%,11.000000,60.000000,67.000000,417.000000
95%,12.000000,71.000000,79.000000,497.700000
99%,14.000000,82.740000,91.000000,557.440000


## 22. Identify empty and very short records


In [ ]:
empty_text_count = alerts_df["combined_text"].str.strip().eq("").sum()
very_short_count = alerts_df["combined_word_count"].lt(10).sum()

print("Empty combined-text records:", empty_text_count)
print("Records with fewer than 10 words:", very_short_count)

short_alert_columns = [
    column for column in
    ["id", "parkCode", "category", "title", "description", "combined_word_count"]
    if column in alerts_df.columns
]

display(
    alerts_df.loc[
        alerts_df["combined_word_count"].lt(10),
        short_alert_columns
    ].sort_values("combined_word_count").head(20)
)


Empty combined-text records: 0
Records with fewer than 10 words: 2


,id,parkCode,category,title,description,combined_word_count
621,029622AA-F8C4-40E2-99C1-13479E55D4ED,inau,Danger,Test Alert,Test Alert,4
246,2F801986-6885-4AD0-B868-1A20A47DAD80,moja,Park Closure,Hart Mine Road Closed,Hart Mine Road Closed.,8


## 23. Collect park names from the NPS parks endpoint

The alert response supplies park codes, but the full park names are helpful for interpretation and held-out-park evaluation.


In [ ]:
PARKS_ENDPOINT = "https://developer.nps.gov/api/v1/parks"

all_parks, reported_park_total = collect_paginated_nps_data(
    endpoint=PARKS_ENDPOINT,
    headers=headers,
    batch_size=50
)

parks_df = pd.DataFrame(all_parks)

print("\nPark DataFrame shape:", parks_df.shape)
print("Park columns:", parks_df.columns.tolist())

park_lookup_columns = [
    column for column in
    ["parkCode", "fullName", "name", "states", "designation"]
    if column in parks_df.columns
]

park_lookup_df = (
    parks_df[park_lookup_columns]
    .drop_duplicates(subset="parkCode")
    .copy()
)

display(park_lookup_df.head())


API reports 474 total records.
Collected 50 of 474 records.
Collected 100 of 474 records.
Collected 150 of 474 records.
Collected 200 of 474 records.
Collected 250 of 474 records.
Collected 300 of 474 records.
Collected 350 of 474 records.
Collected 400 of 474 records.
Collected 450 of 474 records.
Collected 474 of 474 records.

Park DataFrame shape: (474, 25)
Park columns: ['id', 'url', 'fullName', 'parkCode', 'description', 'latitude', 'longitude', 'latLong', 'activities', 'topics', 'states', 'contacts', 'entranceFees', 'entrancePasses', 'fees', 'directionsInfo', 'directionsUrl', 'operatingHours', 'addresses', 'images', 'weatherInfo', 'name', 'designation', 'multimedia', 'relevanceScore']


,parkCode,fullName,name,states,designation
0,abli,Abraham Lincoln Birthplace National Historical...,Abraham Lincoln Birthplace,KY,National Historical Park
1,acad,Acadia National Park,Acadia,ME,National Park
2,adam,Adams National Historical Park,Adams,MA,National Historical Park
3,afam,African American Civil War Memorial,African American Civil War Memorial,DC,
4,afbg,African Burial Ground National Monument,African Burial Ground,NY,National Monument


## 24. Join full park information to the alerts


In [ ]:
alerts_enriched_df = alerts_df.merge(
    park_lookup_df,
    on="parkCode",
    how="left",
    validate="many_to_one"
)

if "fullName" in alerts_enriched_df.columns:
    print(
        "Alerts without a matched full park name:",
        alerts_enriched_df["fullName"].isna().sum()
    )

print("Enriched DataFrame shape:", alerts_enriched_df.shape)
display(alerts_enriched_df.head())


Alerts without a matched full park name: 4
Enriched DataFrame shape: (627, 19)


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate,collection_timestamp_utc,collection_date,combined_text,title_word_count,description_word_count,combined_word_count,combined_character_count,fullName,name,states,designation
0,44ED09F6-52BC-4A67-9EE4-4D92DF0A75FB,,Road Closures 6/24 and 6/25 on Swift Brook and...,kaww,"On Wednesday, 6/24, and Thursday, 6/25, road c...",Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23,Road Closures 6/24 and 6/25 on Swift Brook and...,12,50,62,355,Katahdin Woods and Waters National Monument,Katahdin Woods and Waters,ME,National Monument
1,FEDAF34D-B0F4-4399-9B06-14CBFFE89E42,,Temporary Closure of Pinnacle Road - June 25,cuga,"Pinnacle Road will be closed on Thursday, June...",Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23,Temporary Closure of Pinnacle Road - June 25 P...,8,18,26,145,Cumberland Gap National Historical Park,Cumberland Gap,"KY,TN,VA",National Historical Park
2,8DF8B523-E5C6-4E4C-AF5B-F641F8C3B6EE,https://camba.dualrates.com/a/r/szz/camba/trai...,Current East Rim Trail Status: Open,cuva,"As of Tuesday, June 16, 2026, the East Rim Tra...",Information,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23,Current East Rim Trail Status: Open As of Tues...,6,65,71,411,Cuyahoga Valley National Park,Cuyahoga Valley,OH,National Park
3,F08A2A6E-5846-4829-B869-D51C077B067D,,Multiple Park Road Closures After Heavy Storm ...,vafo,Multiple park roads are closed due to downed t...,Park Closure,[{'title': 'Trees Down Due to Heavy Storm Wind...,2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23,Multiple Park Road Closures After Heavy Storm ...,11,85,96,575,Valley Forge National Historical Park,Valley Forge,PA,National Historical Park
4,336BD9D6-432A-4F8F-A4FC-6EC2CFB285DE,,No Public Beach Access from Fort Moultrie duri...,fosu,There will be no public beach access from Fort...,Park Closure,[],2026-06-23 00:00:00.0,2026-06-23T19:09:17.936120+00:00,2026-06-23,No Public Beach Access from Fort Moultrie duri...,11,22,33,196,Fort Sumter and Fort Moultrie National Histori...,Fort Sumter and Fort Moultrie,SC,National Historical Park


## 25. Inspect the distribution of alerts across parks


In [ ]:
park_name_column = (
    "fullName"
    if "fullName" in alerts_enriched_df.columns
    else "parkCode"
)

alerts_by_park = (
    alerts_enriched_df
    .groupby(["parkCode", park_name_column], dropna=False)
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)

print("Unique park codes represented:", alerts_enriched_df["parkCode"].nunique())
display(alerts_by_park.head(25))


Unique park codes represented: 293


,parkCode,fullName,alert_count
146,iatr,Ice Age National Scenic Trail,10
202,noca,North Cascades National Park,9
122,goga,Golden Gate National Recreation Area,8
187,moja,Mojave National Preserve,7
283,whsa,White Sands National Park,6
172,lavo,Lassen Volcanic National Park,6
22,beol,Bent's Old Fort National Historic Site,6
113,gate,Gateway National Recreation Area,6
272,vafo,Valley Forge National Historical Park,6
213,pagr,Paterson Great Falls National Historical Park,5


## 26. Check exact duplicate text

Duplicate text is not automatically removed here. First, identify duplicate groups and inspect whether they represent repeated records, common templates, or legitimate park-specific alerts.


In [ ]:
normalized_text = (
    alerts_enriched_df["combined_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

alerts_enriched_df["normalized_text"] = normalized_text

exact_duplicate_mask = alerts_enriched_df.duplicated(
    subset="normalized_text",
    keep=False
) & alerts_enriched_df["normalized_text"].ne("")

exact_duplicate_rows = alerts_enriched_df.loc[
    exact_duplicate_mask,
    [
        column for column in
        ["id", "parkCode", park_name_column, "category", "title", "normalized_text"]
        if column in alerts_enriched_df.columns
    ]
].sort_values("normalized_text")

print("Rows belonging to exact duplicate-text groups:", len(exact_duplicate_rows))
print(
    "Unique duplicated text values:",
    exact_duplicate_rows["normalized_text"].nunique()
    if not exact_duplicate_rows.empty else 0
)

display(exact_duplicate_rows.head(30))


Rows belonging to exact duplicate-text groups: 38
Unique duplicated text values: 13


,id,parkCode,fullName,category,title,normalized_text
431,B831CEB1-40B7-455C-9E09-644D8291FC7A,goga,Golden Gate National Recreation Area,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
553,B127187F-76DD-4147-8AB5-CEC00AA6700D,muwo,Muir Woods National Monument,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
554,FDC95683-8853-4829-8A84-6E7E886B1C7A,prsf,Presidio of San Francisco,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
555,B7A06513-5CF2-4CC8-9091-3B80E52C2D0B,fopo,Fort Point National Historic Site,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
483,800F736B-76E9-46C8-AFD9-B756F76EBC32,hamp,Hampton National Historic Site,Information,Baltimore Area Traffic Advisory,baltimore area traffic advisory traffic in the...
485,990FA2AA-0B32-4DA1-9584-7C6DA8CF4EF4,fomc,Fort McHenry National Monument and Historic Sh...,Information,Baltimore Area Traffic Advisory,baltimore area traffic advisory traffic in the...
11,BCB1ABBF-4FA2-4334-916D-4D206017C113,fopo,Fort Point National Historic Site,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
14,8A1F9B2A-17EC-4824-80D7-DFE1616DC4FA,prsf,Presidio of San Francisco,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
15,7A4A704E-7103-48EF-87F5-45D3BB2D733E,goga,Golden Gate National Recreation Area,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
90,6E732798-DC47-424B-8FA8-2FC6E365D3F7,camo,Castle Mountains National Monument,Caution,Fire restrictions in place,fire restrictions in place fire restrictions a...


## 27. Save the enriched working dataset


In [ ]:
enriched_csv_path = (
    output_dir / f"nps_alerts_enriched_{timestamp_for_filename}.csv"
)

alerts_enriched_df.to_csv(enriched_csv_path, index=False)

print("Saved enriched CSV:", enriched_csv_path)


Saved enriched CSV: nps_alert_data/nps_alerts_enriched_20260623T190917Z.csv


# Taxonomy Review and Annotation Sample




## 29. Create review sample for analysis

Remove test records, flag missing park metadata and preserve one row from each exact duplicate-text group for sampling.


In [ ]:
review_pool = alerts_enriched_df.copy()

review_pool["is_obvious_test_record"] = (
    review_pool["title"].fillna("").str.strip().str.lower().eq("test alert")
    & review_pool["description"].fillna("").str.strip().str.lower().eq("test alert")
)

review_pool["missing_park_code"] = (
    review_pool["parkCode"].isna()
    | review_pool["parkCode"].astype(str).str.strip().eq("")
)

review_pool["missing_park_name"] = (
    review_pool["fullName"].isna()
    if "fullName" in review_pool.columns
    else True
)

print("Obvious test records:", review_pool["is_obvious_test_record"].sum())
print("Records missing park code:", review_pool["missing_park_code"].sum())
print("Records missing matched park name:", review_pool["missing_park_name"].sum())

review_pool = review_pool.loc[
    ~review_pool["is_obvious_test_record"]
].copy()

review_pool["duplicate_text_group_size"] = (
    review_pool.groupby("normalized_text")["id"].transform("size")
)

sampling_pool = (
    review_pool
    .sort_values(["normalized_text", "parkCode", "id"])
    .drop_duplicates(subset="normalized_text", keep="first")
    .copy()
)

print("\nReview pool after removing test records:", len(review_pool))
print("Unique-text sampling pool:", len(sampling_pool))
print(
    "Rows set aside from sampling because of exact duplicate text:",
    len(review_pool) - len(sampling_pool)
)


Obvious test records: 1
Records missing park code: 1
Records missing matched park name: 4

Review pool after removing test records: 626
Unique-text sampling pool: 601
Rows set aside from sampling because of exact duplicate text: 25


## 30. Inspect records with missing park metadata



In [ ]:
metadata_issue_columns = [
    column for column in
    ["id", "parkCode", "fullName", "title", "description", "category"]
    if column in review_pool.columns
]

metadata_issues = review_pool.loc[
    review_pool["missing_park_code"] | review_pool["missing_park_name"],
    metadata_issue_columns
]

display(metadata_issues)


,id,parkCode,fullName,title,description,category
475,1377D2C5-6217-411E-98B3-07F75FCA0D3F,es-es/cham,NaN,"Apertura retrasada hoy, 10 de enero debido a m...",Debido a condiciones del tiempo y riesgos elev...,Park Closure
512,687B5063-8BA1-4940-8245-1AB2AC2BDF0A,,NaN,Mansion ADA Lift is currently not functional a...,Mansion ADA Lift is currently not functional a...,Information
613,D2043EA0-DF82-46FB-B3A1-6CECDD141B8D,opot,NaN,Federal Government Shutdown,"During the federal government shutdown, this w...",Park Closure


## 31. Review the original NPS categories after basic exclusions


In [ ]:
review_category_counts = (
    review_pool["category"]
    .value_counts()
    .rename_axis("nps_category")
    .reset_index(name="alert_count")
)

review_category_counts["percent"] = (
    review_category_counts["alert_count"] / len(review_pool) * 100
).round(2)

display(review_category_counts)


,nps_category,alert_count,percent
0,Information,329,52.56
1,Park Closure,154,24.60
2,Caution,132,21.09
3,Danger,11,1.76


## 32. Define the balanced taxonomy-review sample

There are few Danger alerts. Oversample the less common NPS categories. Limit most parks to no more than two selected alerts when possible.


In [ ]:
SAMPLE_SEED = 123

sample_targets = {
    "Information": 40,
    "Park Closure": 35,
    "Caution": 35,
    "Danger": 10
}

def sample_with_park_cap(
    frame,
    n,
    park_column="parkCode",
    max_per_park=2,
    random_state=123
):
    shuffled = frame.sample(frac=1, random_state=random_state).copy()

    selected_indices = []
    park_counts = {}

    for index, row in shuffled.iterrows():
        park_value = row.get(park_column)
        park_key = (
            str(park_value)
            if pd.notna(park_value) and str(park_value).strip()
            else "__MISSING__"
        )

        if park_counts.get(park_key, 0) < max_per_park:
            selected_indices.append(index)
            park_counts[park_key] = park_counts.get(park_key, 0) + 1

        if len(selected_indices) == n:
            break

    if len(selected_indices) < n:
        remaining = shuffled.loc[
            ~shuffled.index.isin(selected_indices)
        ].head(n - len(selected_indices))
        selected_indices.extend(remaining.index.tolist())

    return frame.loc[selected_indices].copy()

sample_parts = []

for offset, (category_name, target_n) in enumerate(sample_targets.items()):
    category_pool = sampling_pool.loc[
        sampling_pool["category"].eq(category_name)
    ].copy()

    actual_n = min(target_n, len(category_pool))

    category_sample = sample_with_park_cap(
        category_pool,
        n=actual_n,
        max_per_park=2,
        random_state=SAMPLE_SEED + offset
    )

    sample_parts.append(category_sample)

taxonomy_sample = (
    pd.concat(sample_parts, ignore_index=True)
    .sample(frac=1, random_state=SAMPLE_SEED)
    .reset_index(drop=True)
)

taxonomy_sample.insert(
    0,
    "review_id",
    [f"R{number:03d}" for number in range(1, len(taxonomy_sample) + 1)]
)

print("Taxonomy-review sample size:", len(taxonomy_sample))
display(taxonomy_sample["category"].value_counts())
print("Unique park codes in sample:", taxonomy_sample["parkCode"].nunique())
print(
    "Largest number selected from one park:",
    taxonomy_sample["parkCode"].value_counts().max()
)


Taxonomy-review sample size: 120


,count
category,
Information,40
Caution,35
Park Closure,35
Danger,10


Unique park codes in sample: 97
Largest number selected from one park: 3


## 33. Annotation taxonomy

Assign one primary topic label and one trip-impact label to each alert.

### Primary topic labels

| Code | Label | Use when the main practical effect is... |
|---|---|---|
| `TRAIL_ACCESS` | Trail or area access restriction | A trail, route, backcountry zone, natural area, or pedestrian area is closed or restricted |
| `ROAD_TRANSPORT` | Road, parking, or transportation issue | A road, parking area, shuttle, ferry, traffic pattern, or vehicle access is affected |
| `ENVIRONMENT_HAZARD` | Weather, fire, smoke, or environmental hazard | Weather, wildfire, smoke, flooding, heat, ice, avalanche, rockfall, air quality, or terrain creates the main concern |
| `WILDLIFE_HAZARD` | Wildlife hazard or wildlife-related restriction | Wildlife activity creates the main risk, precaution, or closure |
| `FACILITY_SERVICE` | Facility, water, campground, or visitor-service issue | A visitor center, campground, restroom, water system, utility, lodging, or other visitor service is affected |
| `MAINTENANCE_INFO` | Construction, maintenance, or general information | The alert is primarily informational or describes maintenance without a more specific traveler consequence |

### Trip-impact labels

| Code | Label | Meaning |
|---|---|---|
| `TRIP_CHANGING` | Trip-changing | A substantial itinerary change is likely |
| `PREPARATION` | Requires preparation | The trip can likely continue with precautions or a smaller adjustment |
| `INFORMATIONAL` | Informational/minimal impact | Little or no meaningful change is likely |

### Main decision rule

Label the alert according to its primary practical effect on the traveler.



## 34. Add annotation columns

Leave uncertain cases marked for later review.


In [ ]:
annotation_columns = {
    "primary_label": "",
    "impact_label": "",
    "secondary_issue": "",
    "uncertain": "",
    "annotation_notes": "",
    "review_status": "not_reviewed"
}

for column_name, default_value in annotation_columns.items():
    taxonomy_sample[column_name] = default_value

annotation_order = [
    "review_id",
    "id",
    "parkCode",
    "fullName",
    "category",
    "title",
    "description",
    "combined_word_count",
    "duplicate_text_group_size",
    "primary_label",
    "impact_label",
    "secondary_issue",
    "uncertain",
    "annotation_notes",
    "review_status"
]

annotation_order = [
    column for column in annotation_order
    if column in taxonomy_sample.columns
]

annotation_df = taxonomy_sample[annotation_order].copy()

display(annotation_df.head())


,review_id,id,parkCode,fullName,category,title,description,combined_word_count,duplicate_text_group_size,primary_label,impact_label,secondary_issue,uncertain,annotation_notes,review_status
0,R001,CA2DB2F7-145C-4C6A-A84D-7289EA0E1FBC,asis,Assateague Island National Seashore,Information,OSV Status,6/8/26 MD open to State Line. VA South OSV ful...,24,1,,,,,,not_reviewed
1,R002,E9F1FADD-2B61-4C0A-ADD7-4F12D508D260,beol,Bent's Old Fort National Historic Site,Danger,Beat the Heat!,Make sure to take weather into consideration w...,27,1,,,,,,not_reviewed
2,R003,AFB62133-8726-41CB-93A5-848C94B0AE2B,keaq,Kenilworth Park & Aquatic Gardens,Danger,Vehicle Weight Limit at Kenilworth Aquatic Gar...,Due to structural conditions at the driveway e...,43,1,,,,,,not_reviewed
3,R004,572BEC6D-166D-4F94-AA23-1E881CA06BB0,chis,Channel Islands National Park,Caution,Western Gulls Nesting on Anacapa Island From A...,"During this time, visitors will encounter seab...",45,1,,,,,,not_reviewed
4,R005,ADD2833A-D9F5-4708-93D5-E727FA392AF7,orca,Oregon Caves National Monument & Preserve,Information,Road work at Oregon Caves,A pavement resurfacing project is taking place...,40,1,,,,,,not_reviewed


## 35. Export the annotation worksheet




In [ ]:
annotation_output_dir = Path("nps_annotation")
annotation_output_dir.mkdir(exist_ok=True)

annotation_csv_path = (
    annotation_output_dir
    / f"nps_taxonomy_review_sample_{timestamp_for_filename}.csv"
)

annotation_df.to_csv(annotation_csv_path, index=False)

print("Saved annotation worksheet:", annotation_csv_path)


Saved annotation worksheet: nps_annotation/nps_taxonomy_review_sample_20260623T190917Z.csv


## 36. Optional consistency checks after annotation

Run these checks to identify invalid or missing labels after filling/reloading.


In [ ]:
VALID_PRIMARY_LABELS = {
    "TRAIL_ACCESS",
    "ROAD_TRANSPORT",
    "ENVIRONMENT_HAZARD",
    "WILDLIFE_HAZARD",
    "FACILITY_SERVICE",
    "MAINTENANCE_INFO"
}

VALID_IMPACT_LABELS = {
    "TRIP_CHANGING",
    "PREPARATION",
    "INFORMATIONAL"
}

def validate_annotations(labeled_df):
    primary_values = labeled_df["primary_label"].fillna("").str.strip()
    impact_values = labeled_df["impact_label"].fillna("").str.strip()

    invalid_primary = labeled_df.loc[
        primary_values.ne("")
        & ~primary_values.isin(VALID_PRIMARY_LABELS)
    ]

    invalid_impact = labeled_df.loc[
        impact_values.ne("")
        & ~impact_values.isin(VALID_IMPACT_LABELS)
    ]

    print("Rows missing primary label:", primary_values.eq("").sum())
    print("Rows missing impact label:", impact_values.eq("").sum())
    print("Rows with invalid primary label:", len(invalid_primary))
    print("Rows with invalid impact label:", len(invalid_impact))

    return invalid_primary, invalid_impact


# labeled_df = pd.read_csv("path_to_completed_annotation_csv.csv")



## Revised pilot annotation rules

### Primary label

Assign one primary_label representing the alert's main practical consequence for a traveler.

### Secondary label

Use secondary_label when a second issue is important.


### Raw text preservation

Keep the original title and description unchanged.


In [ ]:
# Ensure the annotation worksheet contains the revised fields.

required_annotation_columns = [
    "primary_label",
    "secondary_label",
    "impact_label",
    "uncertain",
    "uncertainty_reason",
    "annotation_notes"
]

for column in required_annotation_columns:
    if column not in annotation_df.columns:
        annotation_df[column] = ""

# Preserve raw text explicitly.
if "title_raw" not in annotation_df.columns and "title" in annotation_df.columns:
    annotation_df["title_raw"] = annotation_df["title"]

if "description_raw" not in annotation_df.columns and "description" in annotation_df.columns:
    annotation_df["description_raw"] = annotation_df["description"]

if "combined_text_raw" not in annotation_df.columns:
    title_source = annotation_df.get("title_raw", "").fillna("").astype(str)
    description_source = annotation_df.get("description_raw", "").fillna("").astype(str)
    annotation_df["combined_text_raw"] = (
        title_source.str.strip() + " " + description_source.str.strip()
    ).str.strip()

# Preserve duplicate-group information for leakage control.
if "duplicate_group_id" not in annotation_df.columns:
    if "normalized_text" in annotation_df.columns:
        duplicate_codes, _ = pd.factorize(annotation_df["normalized_text"])
        annotation_df["duplicate_group_id"] = duplicate_codes
    else:
        annotation_df["duplicate_group_id"] = range(len(annotation_df))

display(annotation_df.head())


,review_id,id,parkCode,fullName,category,title,description,combined_word_count,duplicate_text_group_size,primary_label,...,secondary_issue,uncertain,annotation_notes,review_status,secondary_label,uncertainty_reason,title_raw,description_raw,combined_text_raw,duplicate_group_id
0,R001,CA2DB2F7-145C-4C6A-A84D-7289EA0E1FBC,asis,Assateague Island National Seashore,Information,OSV Status,6/8/26 MD open to State Line. VA South OSV ful...,24,1,,...,,,,not_reviewed,,,OSV Status,6/8/26 MD open to State Line. VA South OSV ful...,OSV Status 6/8/26 MD open to State Line. VA So...,0
1,R002,E9F1FADD-2B61-4C0A-ADD7-4F12D508D260,beol,Bent's Old Fort National Historic Site,Danger,Beat the Heat!,Make sure to take weather into consideration w...,27,1,,...,,,,not_reviewed,,,Beat the Heat!,Make sure to take weather into consideration w...,Beat the Heat! Make sure to take weather into ...,1
2,R003,AFB62133-8726-41CB-93A5-848C94B0AE2B,keaq,Kenilworth Park & Aquatic Gardens,Danger,Vehicle Weight Limit at Kenilworth Aquatic Gar...,Due to structural conditions at the driveway e...,43,1,,...,,,,not_reviewed,,,Vehicle Weight Limit at Kenilworth Aquatic Gar...,Due to structural conditions at the driveway e...,Vehicle Weight Limit at Kenilworth Aquatic Gar...,2
3,R004,572BEC6D-166D-4F94-AA23-1E881CA06BB0,chis,Channel Islands National Park,Caution,Western Gulls Nesting on Anacapa Island From A...,"During this time, visitors will encounter seab...",45,1,,...,,,,not_reviewed,,,Western Gulls Nesting on Anacapa Island From A...,"During this time, visitors will encounter seab...",Western Gulls Nesting on Anacapa Island From A...,3
4,R005,ADD2833A-D9F5-4708-93D5-E727FA392AF7,orca,Oregon Caves National Monument & Preserve,Information,Road work at Oregon Caves,A pavement resurfacing project is taking place...,40,1,,...,,,,not_reviewed,,,Road work at Oregon Caves,A pavement resurfacing project is taking place...,Road work at Oregon Caves A pavement resurfaci...,4


## Checkpoint after 30–40 annotations

Before continuing to all 120 review records, calculate percentage requiring a secondary label, percentage marked uncertain, uncertainty reasons, counts by primary label, counts by impact label, and the most common boundary disagreements recorded in notes.

Decide whether any categories should be merged, a category should be split, an explicit Other/uncertain class is actually necessary, or the single-label task is creating too much information loss.


In [ ]:
import pandas as pd

annotation_df = pd.read_excel(
    "/content/nps_taxonomy_annotation_sample_20260623T190917Z.xlsx"
)

print("Loaded shape:", annotation_df.shape)
display(annotation_df.head())

Loaded shape: (120, 20)


,review_id,id,parkCode,fullName,category,title,description,combined_word_count,duplicate_text_group_size,primary_label,impact_label,uncertain,annotation_notes,review_status,secondary_label,uncertainty_reason,title_raw,description_raw,combined_text_raw,duplicate_group_id
0,R001,CA2DB2F7-145C-4C6A-A84D-7289EA0E1FBC,asis,Assateague Island National Seashore,Information,OSV Status,6/8/26 MD open to State Line. VA South OSV ful...,24,1,road_parking_transportation,requires_preparation,0.0,NaN,reviewed,NaN,NaN,OSV Status,6/8/26 MD open to State Line. VA South OSV ful...,OSV Status 6/8/26 MD open to State Line. VA So...,0
1,R002,E9F1FADD-2B61-4C0A-ADD7-4F12D508D260,beol,Bent's Old Fort National Historic Site,Danger,Beat the Heat!,Make sure to take weather into consideration w...,27,1,weather_fire_environmental_hazard,requires_preparation,0.0,NaN,reviewed,NaN,NaN,Beat the Heat!,Make sure to take weather into consideration w...,Beat the Heat! Make sure to take weather into ...,1
2,R003,AFB62133-8726-41CB-93A5-848C94B0AE2B,keaq,Kenilworth Park & Aquatic Gardens,Danger,Vehicle Weight Limit at Kenilworth Aquatic Gar...,Due to structural conditions at the driveway e...,43,1,road_parking_transportation,requires_preparation,0.0,NaN,reviewed,NaN,NaN,Vehicle Weight Limit at Kenilworth Aquatic Gar...,Due to structural conditions at the driveway e...,Vehicle Weight Limit at Kenilworth Aquatic Gar...,2
3,R004,572BEC6D-166D-4F94-AA23-1E881CA06BB0,chis,Channel Islands National Park,Caution,Western Gulls Nesting on Anacapa Island From A...,"During this time, visitors will encounter seab...",45,1,wildlife_hazard_or_restriction,requires_preparation,0.0,NaN,reviewed,NaN,NaN,Western Gulls Nesting on Anacapa Island From A...,"During this time, visitors will encounter seab...",Western Gulls Nesting on Anacapa Island From A...,3
4,R005,ADD2833A-D9F5-4708-93D5-E727FA392AF7,orca,Oregon Caves National Monument & Preserve,Information,Road work at Oregon Caves,A pavement resurfacing project is taking place...,40,1,road_parking_transportation,requires_preparation,0.0,NaN,reviewed,NaN,NaN,Road work at Oregon Caves,A pavement resurfacing project is taking place...,Road work at Oregon Caves A pavement resurfaci...,4


In [ ]:
print(annotation_df.columns.tolist())

['review_id', 'id', 'parkCode', 'fullName', 'category', 'title', 'description', 'combined_word_count', 'duplicate_text_group_size', 'primary_label', 'impact_label', 'uncertain', 'annotation_notes', 'review_status', 'secondary_label', 'uncertainty_reason', 'title_raw', 'description_raw', 'combined_text_raw', 'duplicate_group_id']


In [ ]:
completed_count = (
    annotation_df["primary_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("Completed annotations:", completed_count)

Completed annotations: 39


In [ ]:
# Run after at least 30 annotations are complete.

completed_mask = annotation_df["primary_label"].astype(str).str.strip().ne("")
pilot_df = annotation_df.loc[completed_mask].copy()

print("Completed annotations:", len(pilot_df))

if len(pilot_df) > 0:
    secondary_rate = (
        pilot_df["secondary_label"].astype(str).str.strip().ne("").mean() * 100
    )
    uncertain_numeric = pd.to_numeric(
        pilot_df["uncertain"], errors="coerce"
    ).fillna(0)
    uncertain_rate = uncertain_numeric.eq(1).mean() * 100

    print(f"Secondary-label rate: {secondary_rate:.1f}%")
    print(f"Uncertain rate: {uncertain_rate:.1f}%")

    print("\nPrimary-label counts:")
    display(pilot_df["primary_label"].value_counts(dropna=False).to_frame("count"))

    print("\nImpact-label counts:")
    display(pilot_df["impact_label"].value_counts(dropna=False).to_frame("count"))

    print("\nUncertainty reasons:")
    display(
        pilot_df.loc[
            uncertain_numeric.eq(1),
            "uncertainty_reason"
        ].value_counts(dropna=False).to_frame("count")
    )
else:
    print("Complete the first 30–40 annotations before using this checkpoint.")


Completed annotations: 120
Secondary-label rate: 100.0%
Uncertain rate: 0.0%

Primary-label counts:


,count
primary_label,
NaN,81
road_parking_transportation,12
trail_or_area_access,10
construction_maintenance_general,7
weather_fire_environmental_hazard,7
facility_water_campground_service,2
wildlife_hazard_or_restriction,1



Impact-label counts:


,count
impact_label,
NaN,81
requires_preparation,23
trip_changing,10
informational_minimal_impact,6



Uncertainty reasons:


,count
uncertainty_reason,


In [ ]:
pilot_df = annotation_df.loc[
    annotation_df["primary_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
].copy()

print("Pilot annotations:", len(pilot_df))

# Uncertainty
uncertain_values = (
    pilot_df["uncertain"]
    .fillna(0)
    .astype(str)
    .str.strip()
    .str.lower()
)

uncertain_mask = uncertain_values.isin(
    ["1", "true", "yes", "y", "uncertain"]
)

print(
    f"Uncertain annotations: {uncertain_mask.sum()} "
    f"({uncertain_mask.mean() * 100:.1f}%)"
)

# Secondary labels
secondary_mask = (
    pilot_df["secondary_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print(
    f"Secondary labels used: {secondary_mask.sum()} "
    f"({secondary_mask.mean() * 100:.1f}%)"
)

print("\nUncertainty reasons:")
display(
    pilot_df.loc[uncertain_mask, "uncertainty_reason"]
    .fillna("Not recorded")
    .value_counts()
    .to_frame("count")
)

print("\nSecondary-label counts:")
display(
    pilot_df.loc[secondary_mask, "secondary_label"]
    .value_counts()
    .to_frame("count")
)

Pilot annotations: 39
Uncertain annotations: 0 (0.0%)
Secondary labels used: 8 (20.5%)

Uncertainty reasons:


,count
uncertainty_reason,



Secondary-label counts:


,count
secondary_label,
trail_or_area_access,3
facility_water_campground_service,1
construction_maintenance_general,1
road_parking_transportation,1
weather_fire_environmental_hazard,1
wildlife_hazard_or_restriction,1


In [ ]:
columns_to_show = [
    "review_id",
    "title",
    "description",
    "primary_label",
    "secondary_label",
    "impact_label",
    "uncertain",
    "annotation_notes",
]

columns_to_show = [
    col for col in columns_to_show
    if col in pilot_df.columns
]

display(
    pilot_df.loc[
        pilot_df["primary_label"]
        == "construction_maintenance_general",
        columns_to_show
    ]
)

,review_id,title,description,primary_label,secondary_label,impact_label,uncertain,annotation_notes
8,R009,Control Tower Closed,The Control Tower is closed due to a restorati...,construction_maintenance_general,NaN,informational_minimal_impact,0.0,NaN
15,R016,Sidewalk walk and parking lot repairs at Hopew...,Hopewell Mound Group is currently experiencing...,construction_maintenance_general,NaN,informational_minimal_impact,0.0,NaN
17,R018,Sections of moat wall closed until Summer 2026,Sections of the moat wall that were damaged by...,construction_maintenance_general,NaN,informational_minimal_impact,0.0,NaN
26,R027,Stamp Status,"Unfortunately, since we are not an official pa...",construction_maintenance_general,NaN,informational_minimal_impact,0.0,NaN
30,R031,Preservation Work Along Trails,"Starting March 18, preservationists restoring ...",construction_maintenance_general,NaN,requires_preparation,0.0,NaN
31,R032,Fort Moultrie Fishing Dock Closure,The Fort Moultrie Fishing Dock is currently un...,construction_maintenance_general,NaN,requires_preparation,0.0,NaN
35,R036,Greenbush Segment — Logging Activities,A timber harvest is taking place on the Greenb...,construction_maintenance_general,weather_fire_environmental_hazard,requires_preparation,0.0,NaN


In [ ]:
unannotated_df = annotation_df.loc[
    annotation_df["primary_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
].copy()

wildlife_terms = (
    r"\bbear\b|\bbison\b|\bwildlife\b|\bnesting\b|"
    r"\banimal\b|\bsnake\b|\bshark\b|\bplover\b"
)

facility_terms = (
    r"visitor center|campground|restroom|water|sanitation|"
    r"electricity|utility|lodging|facility|phone"
)

print("Potential wildlife records:")
display(
    unannotated_df.loc[
        unannotated_df["combined_text_raw"]
        .fillna("")
        .str.contains(wildlife_terms, case=False, regex=True),
        ["review_id", "title", "description", "category"]
    ].head(15)
)

print("\nPotential facility/service records:")
display(
    unannotated_df.loc[
        unannotated_df["combined_text_raw"]
        .fillna("")
        .str.contains(facility_terms, case=False, regex=True),
        ["review_id", "title", "description", "category"]
    ].head(15)
)

Potential wildlife records:


,review_id,title,description,category
40,R041,Temporary Closures for Peregrine Nesting,The following established climbing management ...,Park Closure
43,R044,Campground Closure - Goodell Creek Campground,Goodell Creek Campground is closed until furth...,Park Closure
104,R105,North Beach Tip at Sandy Hook closed,North Beach Tip at Sandy Hook is closed to acc...,Park Closure
108,R109,Elk with Calves are Extremely Dangerous,"Be alert, especially this time of year around ...",Caution
110,R111,Wildlife Viewing Guidelines,San Juan Island National Historical Park provi...,Caution



Potential facility/service records:


,review_id,title,description,category
41,R042,"South Woolum Road Closed, Alternate Route Avai...","In the interest of safety, the South Woolum Ro...",Park Closure
42,R043,Canine distemper is believed to be present in ...,Contact with saliva from wild animals can tran...,Caution
43,R044,Campground Closure - Goodell Creek Campground,Goodell Creek Campground is closed until furth...,Park Closure
44,R045,Harmful Algae Bloom,A Harmful Algae Bloom has been confirmed in re...,Danger
53,R054,Upper and Lower Plaza Rehabilitation,"The Plazas, Rotunda, and the Observation Deck ...",Park Closure
66,R067,Bacteria Levels at Arizona Hot Springs,Fecal bacteria in water samples taken from Ari...,Caution
69,R070,Phone Service Unavailable,Phone service at Puʻukoholā Heiau National His...,Information
70,R071,Intermittent Road Closures During Utility Impr...,"Beginning May 11, Stones River National Battle...",Information
72,R073,Hours Changing,The Visitor Center at Friendship Hill National...,Information
73,R074,Hawk Creek Area Closed for 2026 Recreation Season,The Hawk Creek Campground and boat launch area...,Park Closure


## Export the revised annotation worksheet

This export preserves raw text, duplicate-group information, secondary labels, and uncertainty documentation.


In [ ]:
updated_annotation_path = (
    output_dir / f"nps_taxonomy_annotation_sample_UPDATED_{timestamp_for_filename}.csv"
)

annotation_df.to_csv(updated_annotation_path, index=False)

print("Saved updated annotation worksheet:", updated_annotation_path)


Saved updated annotation worksheet: nps_alert_data/nps_taxonomy_annotation_sample_UPDATED_20260706T145021Z.csv
